# Agentic RAG explainer and demo

## PART 2: Let's go Agentic

In [1]:
from dotenv import load_dotenv
import json
from chromadb import PersistentClient
from openai import OpenAI
from sklearn.manifold import TSNE
import numpy as np
import plotly.graph_objects as go
from agents import Agent, Runner, function_tool

MODEL_NAME = 'gpt-5.4-mini'
DB_NAME = 'vectorstore'
collection_name = 'facts'
embedding_model = "text-embedding-3-small"

load_dotenv(override=True)

True

## Same setup as before

In [2]:
openai = OpenAI()
chroma = PersistentClient(path=DB_NAME)
collection = chroma.get_or_create_collection(collection_name)
print(f"Vectorstore has {collection.count()} documents")

Vectorstore has 36 documents


## Here's a function that does the vector-based retrieval

In [3]:
def retrieve_relevant_context(query):
    query_embedding = openai.embeddings.create(model=embedding_model, input=[query]).data[0].embedding
    results = collection.query(query_embeddings=[query_embedding], n_results=3)
    print(f"Retrieved {len(results['documents'])} relevant documents for the query: '{query}'")
    return "\n".join(results["documents"][0])

In [4]:
retrieve_relevant_context("What is Ed's favorite fruit?")

Retrieved 1 relevant documents for the query: 'What is Ed's favorite fruit?'


"Ed's favorite fruit is a banana\nEd goes by Ed or Edward, whichever you prefer\nEd can be reached via email at ed@edwarddonner.com or on Twitter/X at @edwarddonner or YouTube at youtube.com/@edward.donner"

## OK, let's turn it into a Tool

In [5]:
@function_tool
def retrieve_relevant_context_tool(query: str) -> str:
    """Retrieves relevant context for a given query.

    Args:
        query: The query to retrieve context for."""
    print(f"TOOL CALLED: Retrieving relevant context for query: {query}")
    return retrieve_relevant_context(query)

In [6]:
retrieve_relevant_context_tool.description

'Retrieves relevant context for a given query.'

In [7]:
retrieve_relevant_context_tool.params_json_schema

{'properties': {'query': {'description': 'The query to retrieve context for.',
   'title': 'Query',
   'type': 'string'}},
 'required': ['query'],
 'title': 'retrieve_relevant_context_tool_args',
 'type': 'object',
 'additionalProperties': False}

## And now let's try it out

In [8]:
question = "What is Ed's favorite fruit?"

In [12]:
agent = Agent("Expert", model=MODEL_NAME, tools=[retrieve_relevant_context_tool])
response = await Runner.run(agent, question)
print(response.final_output)

TOOL CALLED: Retrieving relevant context for query: Ed's favorite fruit
Retrieved 1 relevant documents for the query: 'Ed's favorite fruit'
Ed’s favorite fruit is a banana.


## And now - behold - Agentic RAG!

In [ ]:
basic_agentic_rag = Agent("Expert", model=MODEL_NAME, tools=[retrieve_relevant_context_tool])
response = await Runner.run(basic_agentic_rag, question)
print(response.final_output)

## Getting more sophisticated

Separate out the Retriever into a different Agent

In [13]:
instructions = """
You are a Retrieval Agent.
You are responsible for researching a question, using your tools to find relevant context that will help answer the question.
Do not actually answer the question; instead, use the tools to respond with a summary of relevant context.
"""
retrieval_agent = Agent("Retriever", model=MODEL_NAME, instructions=instructions, tools=[retrieve_relevant_context_tool])
response = await Runner.run(retrieval_agent, question)
print(response.final_output)

TOOL CALLED: Retrieving relevant context for query: Ed's favorite fruit
Retrieved 1 relevant documents for the query: 'Ed's favorite fruit'
Relevant context found: Ed’s favorite fruit is a banana.


In [14]:
retriever = retrieval_agent.as_tool(tool_name="retriever", tool_description="Use this tool to retrieve relevant context for a question that you provide")

In [15]:
full_agentic_rag = Agent("Expert", model=MODEL_NAME, tools=[retriever])
response = await Runner.run(full_agentic_rag, question)
print(response.final_output)

TOOL CALLED: Retrieving relevant context for query: Ed and his favorite fruit
Retrieved 1 relevant documents for the query: 'Ed and his favorite fruit'
Ed’s favorite fruit is a banana.


## Taking this further

Continue this approach to add more tools to the Retriever Agent:
- Ability to lookup across different sources: eg SQL access, Graph database, Web search
- Ability to use different kinds of "memory"
- Keep retrieving and synthesizing until it is satisfied

## Think of it like progressing your Agent from being a Retriever to being a Researcher

"Agentic RAG" becomes just a part of your overall Agent architecture.

## How to decide which approach to use?

Always be driven off real-world evaluations. Set a metric, run experiments, pick the approach that performs best for your use case.